In [ ]:
import os
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim

import torch.utils.data as data
from torch.utils.tensorboard import SummaryWriter

import torchvision
from torchvision import transforms
from torchvision.utils import make_grid
from torchvision.datasets import OxfordIIITPet, CIFAR10

import matplotlib.pyplot as plt
#from tqdm import tqdm
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.stats import norm
import json
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import torchinfo
# pip install umap-learn
import umap.umap_ as umap

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap
import numpy as np
import torch
import numpy as np
import scipy.stats as stats
from sklearn.neighbors import NearestNeighbors
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from sklearn.neighbors import NearestNeighbors

from sklearn.neighbors import NearestNeighbors
import seaborn as sns
from pathlib import Path
from datetime import datetime


In [ ]:
# 1. Setup Base Paths
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

data_dir = Path("data")
base_dir = Path("generative")
run_dir  = base_dir / f"run_{timestamp}"

models = ["ae", "vae"]
dirs = {}

# 2. Define Hierarchy
for model in models:
    model_dir = run_dir / model
    dirs[model] = {
        "root": model_dir,
        "logs": model_dir / "logs",
        "checkpoints": model_dir / "checkpoints",
        "configs": model_dir / "configs",
        "results": model_dir / "results",
    }

# 3. Create folders
data_dir.mkdir(parents=True, exist_ok=True)

for model_paths in dirs.values():
    for path in model_paths.values():
        path.mkdir(parents=True, exist_ok=True)

# Example Usage:
print(f"AE Logs will be saved to: {dirs['ae']['logs']}")


In [ ]:
import torch
import numpy as np
import random
import os

def seed_everything(seed=42):
    # 1. Basic Python and OS Level
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
    # 2. Torch CPU
    torch.manual_seed(seed)
    
    # 3. NVIDIA CUDA (Nvidia GPUs)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # for multi-GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    # 4. Apple MPS (M1/M2/M3 Macs)
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
        # Note: torch.use_deterministic_algorithms(True) often fails on MPS
        # because Metal shaders don't support it for all ops yet.
    
    # 5. Global Determinism (Optional/Strict)
    # This will throw an error if an op doesn't have a deterministic impl.
    # torch.use_deterministic_algorithms(True, warn_only=True)


In [ ]:
seed_everything(42)

In [ ]:
def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")
    

In [ ]:
device = get_device()
print(device)

In [ ]:
# Transformations applied on each image => only make them a tensor
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

# Loading the training dataset. We need to split it into a training and validation part
cifar10_train_dataset = CIFAR10(root=data_dir, train=True,  transform=transform, download=True)
# Loading the test set
cifar10_test_set      = CIFAR10(root=data_dir, train=False, transform=transform, download=True)

cifar10_train_set, cifar10_val_set = torch.utils.data.random_split(cifar10_train_dataset, [45000, 5000])

batch_size = 256
pin_memory = get_device() == "cuda"
num_workers = 0
# We define a set of data loaders that we can use for various purposes later.

cifar10_train_loader = data.DataLoader(cifar10_train_set, batch_size=batch_size, 
                                       shuffle=True,  drop_last=True, 
                                       pin_memory=pin_memory, num_workers=num_workers)

cifar10_val_loader   = data.DataLoader(cifar10_val_set, batch_size=batch_size, 
                                       shuffle=False, drop_last=False, 
                                       pin_memory=pin_memory, num_workers=num_workers)

cifar10_test_loader  = data.DataLoader(cifar10_test_set, batch_size=batch_size, 
                                       shuffle=False, drop_last=False, 
                                       pin_memory=pin_memory, num_workers=num_workers)



In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = np.inf
        self.best_rec = np.inf
        self.best_kld = np.inf
        self.early_stop = False
        self.best_state = None
        self.best_epoch = -1

    def __call__(self, model, epoch, val_loss, val_rec, val_kld):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_rec = val_rec
            self.best_kld = val_kld
            self.best_epoch = epoch
            self.counter = 0

            self.best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

    def load_best_weights(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)
            print(f"Restored best model weights (Val Loss: {self.best_loss:.4f})")


In [ ]:
class AE_Encoder(nn.Module):
    def __init__(self, num_input_channels=3, base_channel_size=32, latent_dim=64):
        super().__init__()
        # 32x32 -> 16x16
        self.latent_dim = latent_dim
        self.block1 = nn.Sequential(
            nn.Conv2d(num_input_channels, base_channel_size, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size),
            nn.GELU(),
            nn.Conv2d(base_channel_size,  base_channel_size, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size),
            nn.GELU(),
        )

        # 16x16 -> 8x8
        self.block2 = nn.Sequential(
            nn.Conv2d(base_channel_size,     base_channel_size * 2, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 2),
            nn.GELU(),
            nn.Conv2d(base_channel_size * 2, base_channel_size * 2, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 2),
            nn.GELU(),
        )

        # 8x8 -> 4x4
        self.block3 = nn.Sequential(
            nn.Conv2d(base_channel_size * 2, base_channel_size * 4, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 4),
            nn.GELU(),
            nn.Conv2d(base_channel_size * 4, base_channel_size * 4, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 4),
            nn.GELU(),
        )

        # latent projection
        self.fc = nn.Linear(base_channel_size * 4 * 4 * 4, latent_dim)

    def forward(self, x):
        x = self.block1(x)   # -> B x C x 16 x 16
        x = self.block2(x)   # -> B x 2C x 8 x 8
        x = self.block3(x)   # -> B x 4C x 4 x 4
        x = torch.flatten(x, 1)
        z = self.fc(x)
        return z, None



class AE_Decoder(nn.Module):
    def __init__(self, num_input_channels=3, base_channel_size=32, latent_dim=64):
        super().__init__()
        # latent -> 4x4 feature map
        self.latent_dim = latent_dim
        self.fc = nn.Linear(latent_dim, base_channel_size * 4 * 4 * 4)

        # 4x4 -> 8x8
        self.block1 = nn.Sequential(
            nn.Conv2d(base_channel_size * 4, base_channel_size * 4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 4),
            nn.GELU(),
            nn.Conv2d(base_channel_size * 4, base_channel_size * 2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 2),
            nn.GELU(),
        )

        # 8x8 -> 16x16
        self.block2 = nn.Sequential(
            nn.Conv2d(base_channel_size * 2, base_channel_size * 2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 2),
            nn.GELU(),
            nn.Conv2d(base_channel_size * 2, base_channel_size,     kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size),
            nn.GELU(),
        )

        # 16x16 -> 32x32
        self.block3 = nn.Sequential(
            nn.Conv2d(base_channel_size, base_channel_size,   kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size),
            nn.GELU(),
            nn.Conv2d(base_channel_size, num_input_channels, kernel_size=3, padding=1, bias=True),
        )

        # Use Tanh for normalized images in [-1,1]
        self.output_act = nn.Tanh()

    def forward(self, z):
        x = self.fc(z)
        x = x.view(z.size(0), -1, 4, 4)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.block1(x)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.block2(x)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.block3(x)
        return self.output_act(x)


class AE(nn.Module):
    def __init__(self, num_input_channels=3, base_channel_size=32, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = AE_Encoder(
            num_input_channels=num_input_channels,
            base_channel_size=base_channel_size,
            latent_dim=latent_dim
        )
        self.decoder = AE_Decoder(
            num_input_channels=num_input_channels,
            base_channel_size=base_channel_size,
            latent_dim=latent_dim
        )

    def forward(self, x):
        z, _ = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, None, None


In [ ]:
def gn(channels, max_groups=8):
    groups = min(max_groups, channels)
    while channels % groups != 0:
        groups -= 1
    return nn.GroupNorm(groups, channels)

class VAE_Encoder(nn.Module):
    def __init__(self, num_input_channels=3, base_channel_size=32, latent_dim=64):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(num_input_channels, base_channel_size, 3, stride=1, padding=1, bias=False),
            gn(base_channel_size),
            #nn.BatchNorm2d(base_channel_size),
            nn.GELU(),
            nn.Conv2d(base_channel_size,  base_channel_size, 3, stride=2, padding=1, bias=False),
            gn(base_channel_size),
            #nn.BatchNorm2d(base_channel_size),
            nn.GELU(),
        )  # 32 -> 16

        self.block2 = nn.Sequential(
            nn.Conv2d(base_channel_size, base_channel_size * 2, 3,     stride=1, padding=1, bias=False),
            gn(base_channel_size * 2),
            #nn.BatchNorm2d(base_channel_size*2),
            nn.GELU(),
            nn.Conv2d(base_channel_size * 2, base_channel_size * 2, 3, stride=2, padding=1, bias=False),
            gn(base_channel_size * 2),
            #nn.BatchNorm2d(base_channel_size*2),
            nn.GELU(),
        )  # 16 -> 8

        self.block3 = nn.Sequential(
            nn.Conv2d(base_channel_size * 2, base_channel_size * 4, 3, stride=1, padding=1, bias=False),
            gn(base_channel_size * 4),
            #nn.BatchNorm2d(base_channel_size*4),
            nn.GELU(),
            nn.Conv2d(base_channel_size * 4, base_channel_size * 4, 3, stride=2, padding=1, bias=False),
            gn(base_channel_size * 4),
            #nn.BatchNorm2d(base_channel_size*4),
            nn.GELU(),
        )  # 8 -> 4
        flat_features = base_channel_size * 4 * 4 * 4 
        self.fc_mu = nn.Linear(flat_features, latent_dim)
        self.fc_logvar = nn.Linear(flat_features, latent_dim)


    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = torch.flatten(x, 1)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar


class VAE_Decoder(nn.Module):
    def __init__(self, num_input_channels=3, base_channel_size=32, latent_dim=64):
        super().__init__()
        self.fc = nn.Linear(latent_dim, base_channel_size * 4 * 4 * 4)
        self.block1 = nn.Sequential(
            nn.Conv2d(base_channel_size * 4, base_channel_size * 4, 3, padding=1, stride=1, bias=False),
            gn(base_channel_size * 4),
            #nn.BatchNorm2d(base_channel_size*4),
            nn.GELU(),
            nn.Conv2d(base_channel_size * 4, base_channel_size * 2, 3, padding=1, stride=1, bias=False),
            gn(base_channel_size * 2),
            #nn.BatchNorm2d(base_channel_size*2),
            nn.GELU(),
        )  # 4 -> 8

        self.block2 = nn.Sequential(
            nn.Conv2d(base_channel_size * 2, base_channel_size * 2, 3, padding=1, stride=1, bias=False),
            gn(base_channel_size * 2),
            #nn.BatchNorm2d(base_channel_size*2),
            nn.GELU(),
            nn.Conv2d(base_channel_size * 2, base_channel_size, 3, padding=1, stride=1, bias=False),
            gn(base_channel_size),
            #nn.BatchNorm2d(base_channel_size),
            nn.GELU(),
        )  # 8 -> 16

        self.block3 = nn.Sequential(
            nn.Conv2d(base_channel_size, base_channel_size, 3, padding=1, stride=1, bias=False),
            gn(base_channel_size),
            #nn.BatchNorm2d(base_channel_size),
            nn.GELU(),
            nn.Conv2d(base_channel_size, num_input_channels, 3, padding=1, stride=1, bias=True),
        )  # 16 -> 32

        self.output_act = nn.Tanh()
        # use Sigmoid() instead if images are in [0,1]

    def forward(self, z):
        x = self.fc(z)
        x = x.view(z.size(0), -1, 4, 4)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.block1(x)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.block2(x)
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.block3(x)
        return self.output_act(x)


class VAE(nn.Module):
    def __init__(self, num_input_channels=3, base_channel_size=32, latent_dim=64):
        super().__init__()
        self.name = "vae"
        self.latent_dim = latent_dim
        self.encoder = VAE_Encoder(
            num_input_channels=num_input_channels,
            base_channel_size=base_channel_size,
            latent_dim=latent_dim
        )

        self.decoder = VAE_Decoder(
            num_input_channels=num_input_channels,
            base_channel_size=base_channel_size,
            latent_dim=latent_dim
        )

    def reparameterize(self, mu, logvar):
        """
        z = mu + std * eps
        std = exp(0.5 * logvar)
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar


In [ ]:
def reconstruction_loss(y_pred, y_true):
    loss = F.mse_loss(y_pred, y_true, reduction='none')
    loss = loss.view(loss.size(0), -1).sum(dim=1)
    return loss.mean(dim=0)

def kl_divergence(mu, log_var):
        # Closed-form solution for KL divergence of a Gaussian
        loss = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp())
        loss = loss.sum(dim=1)
        return loss.mean(dim=0)
    
class AECriterion:
    def __init__(self, beta=0.):
        self.beta = 0

    def __call__(self, x_hat, x, mu=None, log_var=None, beta=None):
        rec = reconstruction_loss(x_hat, x) 
        # Return format: total, rec, kld (None for AE)
        return rec, rec, torch.tensor(0.0, device=x.device)

class VAECriterion:
    def __init__(self, beta=0.5):
        self.beta = beta

    def __call__(self, x_hat, x, mu, log_var):
        rec = reconstruction_loss(x_hat, x) 
        kld = kl_divergence(mu, log_var)
        return rec + (self.beta * kld), rec, kld


In [ ]:
#def get_optimizer(model):
#    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
#    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#        optimizer, mode="min", factor=0.5, patience=10, min_lr=1e-6)
#    return optimizer, scheduler

def get_AdamW_CosineAnnealingLR(model, lr, weight_decay, T_max, eta_min):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_max, eta_min=eta_min)
    return optimizer, scheduler


In [ ]:
def train(train_loader, val_loader, test_loader, 
          model, criterion, 
          optimizer, scheduler, 
          max_epochs, early_stopping,
          writer, checkpoint_filename, result_filename): 
    
    device = next(model.parameters()).device
    print(f"Training on {device}")
    
    for epoch in range(max_epochs):
        # -------- TRAIN --------
        model.train()
        train_loss_sum = 0.0
        train_rec_sum  = 0.0
        train_kld_sum  = 0.0
        train_num_samples = 0
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch:03d} [Train]")
        for x, _ in train_bar:
            batch_size = x.size(0)
            train_num_samples += batch_size
            x = x.to(device)
            
            optimizer.zero_grad(set_to_none=True)
            x_hat, mu, log_var = model(x)
            loss, rec, kld = criterion(x_hat, x, mu, log_var)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            
            train_loss_sum += loss.item() * batch_size
            train_rec_sum  += rec.item()  * batch_size
            train_kld_sum  += kld.item()  * batch_size
            #if kld is not None:
            #    train_kld_sum += kld.item() * batch_size
            
            train_bar.set_postfix(loss=loss.item())

        train_loss = train_loss_sum / train_num_samples
        train_rec  = train_rec_sum  / train_num_samples
        train_kld  = train_kld_sum  / train_num_samples

        # -------- VALIDATION --------
        model.eval()
        val_loss_sum = 0.0
        val_rec_sum = 0.0
        val_kld_sum = 0.0
        val_num_samples = 0
        
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch:03d} [Val]")
        with torch.no_grad():
            for x, _ in val_bar:
                x = x.to(device)
                batch_size = x.size(0)
                val_num_samples += batch_size
                x_hat, mu, log_var = model(x)
                loss, rec, kld = criterion(x_hat, x, mu, log_var)
                val_loss_sum += loss.item() * batch_size
                val_rec_sum  += rec.item()  * batch_size
                val_kld_sum  += kld.item()  * batch_size
                val_bar.set_postfix(loss=loss.item())
        
        val_loss = val_loss_sum / val_num_samples
        val_rec  = val_rec_sum  / val_num_samples
        val_kld  = val_kld_sum  / val_num_samples

        # -------------------- SCHEDULER --------------------
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_loss)
        else:
            scheduler.step()

        # -------------------- LOGGING --------------------
        writer.add_scalar("loss/train", train_loss, epoch)
        writer.add_scalar("loss/val",   val_loss, epoch)
        writer.add_scalar("rec/train",  train_rec, epoch)
        writer.add_scalar("rec/val",    val_rec, epoch)
        writer.add_scalar("kld/train",  train_kld, epoch)
        writer.add_scalar("kld/val",    val_kld, epoch)
        writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)

        print(
            f"Epoch {epoch:03d} | "
            f"Train: Loss = {train_loss:.4f} Rec = {train_rec:.4f} KLD = {train_kld:.4f} | "
            f"Val: Loss = {val_loss:.4f} Rec = {val_rec:.4f} KLD = {val_kld:.4f}"
        )
        
        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                # Sample from standard normal
                samples = torch.randn(16, model.latent_dim).to(device)
                gen_images = model.decoder(samples).cpu()
                grid = torchvision.utils.make_grid(gen_images, normalize=True)
                writer.add_image("samples", grid, epoch)
        
        # -------------------- EARLY STOPPING --------------------

        if early_stopping(model, epoch, val_loss, val_rec, val_kld):
            print("Stopping early")
            break
            
    # Before saving or testing, revert to the best discovered state
    early_stopping.load_best_weights(model)
    ## -------------------- SAVE --------------------
    #torch.save(
    #    model.state_dict(),
    #    os.path.join(checkpoint_path, f"{model.name}_{model.latent_dim}.pt")
    #)
    torch.save({
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_epoch": early_stopping.best_epoch,
    }, checkpoint_filename)
    # -------------------- TEST --------------------
    model.eval()
    test_loss_sum = 0.0
    test_rec_sum = 0.0
    test_kld_sum = 0.0
    test_num_samples = 0
    with torch.no_grad():
        for x, _ in test_loader:
            batch_size = x.size(0)
            test_num_samples += batch_size
            x = x.to(device)
            x_hat, mu, log_var = model(x)
            loss, rec, kld = criterion(x_hat, x, mu, log_var)
            test_loss_sum += loss.item() * batch_size
            test_rec_sum  += rec.item()  * batch_size
            if kld is not None:
                test_kld_sum += kld.item() * batch_size
    
    test_loss = test_loss_sum / test_num_samples
    test_rec  = test_rec_sum  / test_num_samples
    test_kld  = test_kld_sum  / test_num_samples
    print(f"Test: Loss = {test_loss:.4f} Rec = {test_rec:.4f} KLD = {test_kld:.4f}")  
    # After the test results are calculated
    
    result = {
        "test": {"loss": test_loss, "rec": test_rec, "kld": test_kld},
        "val":  {"loss": early_stopping.best_loss, 
                 "rec": early_stopping.best_rec,
                 "kld": early_stopping.best_kld,
                 "best_epoch": early_stopping.best_epoch,
                },
    }
    # Save results next to the checkpoint
    with open(result_filename, "w") as f:
        json.dump(result, f, indent=4)
    
    print(f"Results saved to {result_filename}")
    return result


In [ ]:
MAX_EPOCHS = 100
ae_hparams = {
    "model": {"latent_dim": 64, "num_input_channels": 3, "base_channel_size":32},
    "training": {"max_epochs": MAX_EPOCHS, "device": str(device),}, 
    "opt": {"lr": 1e-3, "weight_decay": 1e-4, "T_max": MAX_EPOCHS, "eta_min": 1e-6, },
    "loss": {"beta": 0.,},
    "early_stopping":{"patience": 15, "min_delta": 0.001},
    "dataloader": {"batch_size": batch_size, "pin_memory":pin_memory, "num_workers":num_workers}
}

with open(dirs['ae']['configs'] / "hyperparams.json", "w") as f:
    json.dump(ae_hparams, f)


In [ ]:
ae_model = AE(**ae_hparams["model"])
print(torchinfo.summary(ae_model, input_size=(batch_size, 3, 32, 32)))
ae_model.to(device)

ae_criterion = AECriterion(**ae_hparams["loss"])
ae_optimizer, ae_scheduler = get_AdamW_CosineAnnealingLR(ae_model, **ae_hparams["opt"])
ae_early_stopping = EarlyStopping(**ae_hparams["early_stopping"])

ae_writer = SummaryWriter(dirs['ae']['logs'])
ae_checkpoint_filename = dirs['ae']['checkpoints'] / "model.pt"
ae_result_filename = dirs['ae']['results'] / "result.json"

In [ ]:
ae_result = train(cifar10_train_loader, cifar10_val_loader, cifar10_test_loader,
                  ae_model, ae_criterion,
                  ae_optimizer, ae_scheduler, 
                  MAX_EPOCHS, ae_early_stopping,
                  ae_writer, ae_checkpoint_filename, ae_result_filename
                 )


In [ ]:
vae_hparams = {
    "model": {"latent_dim": 64, "num_input_channels": 3, "base_channel_size":32},
    "training": {"max_epochs": MAX_EPOCHS, "device": str(device),}, 
    "opt": {"lr": 1e-3, "weight_decay": 1e-4, "T_max": MAX_EPOCHS, "eta_min": 1e-6, },
    "loss": {"beta": 0.5,},
    "early_stopping":{"patience": 15, "min_delta": 0.001},
    "dataloader": {"batch_size": batch_size, "pin_memory":pin_memory, "num_workers":num_workers},
}

with open(dirs['vae']['configs'] / "hyperparams.json", "w") as f:
    json.dump(vae_hparams, f)


In [ ]:
vae_model = VAE(**vae_hparams["model"])
print(torchinfo.summary(vae_model, input_size=(batch_size, 3, 32, 32)))
vae_model.to(device)

vae_criterion = VAECriterion(**vae_hparams["loss"])
vae_optimizer, vae_scheduler = get_AdamW_CosineAnnealingLR(vae_model, **vae_hparams["opt"])
vae_early_stopping = EarlyStopping(**vae_hparams["early_stopping"])

vae_writer = SummaryWriter(dirs['vae']['logs'])
vae_checkpoint_filename = dirs['vae']['checkpoints'] / "model.pt"
vae_result_filename = dirs['vae']['results'] / "result.json"

In [ ]:
vae_result = train(cifar10_train_loader, cifar10_val_loader, cifar10_test_loader,
                  vae_model, vae_criterion,
                  vae_optimizer, vae_scheduler, 
                  MAX_EPOCHS, vae_early_stopping,
                  vae_writer, vae_checkpoint_filename, vae_result_filename
                 )


## Reconstruction Check

This test evaluates whether an Autoencoder (AE) or Variational Autoencoder (VAE) has learned to compress data into a latent bottleneck and reconstruct it meaningfully.

It works by visually comparing **original images** with their **reconstructed versions**.

Reconstruction quality reveals:

- which important features were preserved,
- what information was lost,
- how effective the latent representation is,
- and whether the model learned real structure rather than memorization.

Accurate reconstructions suggest useful feature learning, while blurry or distorted outputs may indicate undertraining, excessive compression, or weak latent representations.

This is often the first and most intuitive diagnostic used to assess Autoencoders.

In [ ]:
def get_train_images(dataset, num):
    return torch.stack([dataset[i][0] for i in range(num)], dim=0)


In [ ]:
def visualize_reconstructions(model, input_imgs, n_samples=8, title_suffix=""):
    """
    Improved reconstruction visualizer.
    - Handles both AE and VAE return types.
    - Limits display to n_samples for better visibility.
    - Automatically manages device transfers.
    """
    model.eval()
    device = next(model.parameters()).device
    
    # 1. Limit the number of images to prevent a giant, unreadable grid
    n_samples = min(len(input_imgs), n_samples)
    test_batch = input_imgs[:n_samples].to(device)

    with torch.no_grad():
        # 2. Flexible unpacking: Handles (recon, mu, var) or just (recon)
        reconst_imgs, _, _ = model(test_batch)
    
    # 3. Create the comparison stack (Original, then Reconstruction)
    # Move to CPU for plotting
    reconst_imgs = reconst_imgs.cpu()
    original_imgs = test_batch.cpu()
    
    comparison_stack = torch.stack([original_imgs, reconst_imgs], dim=0).flatten(0, 1)

    # 4. Grid configuration
    grid = torchvision.utils.make_grid(
        comparison_stack, 
        nrow=len(input_imgs), 
        normalize=True, 
        value_range=(-1, 1),
        padding=2
    )
    
    plt.figure(figsize=(12, n_samples * 1.5)) # Dynamic height based on samples
    plt.imshow(grid.permute(1, 2, 0))
    plt.title(f"Reconstruction Quality {title_suffix}\n(Top: Original | Bottom: Reconstruction)")
    plt.axis("off")
    plt.show()

In [ ]:
input_imgs = get_train_images(cifar10_test_set, 8)
visualize_reconstructions(ae_model, input_imgs, 8, title_suffix="AE")

In [ ]:
visualize_reconstructions(vae_model, input_imgs, 8, title_suffix="VAE")

## Reconstruction vs Regularization Trade-off

When comparing an Autoencoder (AE) and a Variational Autoencoder (VAE), the AE almost always achieves lower reconstruction error. 

This reflects a fundamental trade-off between reconstruction accuracy and latent space structure.

- **AE:** prioritizes reconstruction accuracy. It can encode fine details freely, producing low error but an unstructured latent space. It behaves like a strong compressor but weak generator.  

- **VAE:** balances reconstruction with a KL-divergence constraint that forces the latent space toward a standard Gaussian distribution. This improves structure and generative ability but increases reconstruction error due to regularization.


The VAE is not only minimizing reconstruction loss—it is also shaping a smooth, continuous latent space. 

This "constraint pressure" often leads to slightly blurred or less precise reconstructions, but much better sampling and interpolation behavior.

- **AE < VAE error (normal):** expected trade-off between accuracy and structure  
- **Very large VAE error:** may indicate over-strong KL regularization  
- **Posterior collapse:** VAE ignores input and over-prioritizes Gaussian constraint, losing reconstruction detail  

This comparison highlights the central tension in representation learning:  
precision of reconstruction vs. quality and usability of the latent space.

In [ ]:
def test_reconstruction_fidelity(model, dataloader, title_suffix=""):
    """
    Improved Reconstruction Fidelity Test.
    - Handles AE/VAE return variations.
    - Uses weighted averaging for varied batch sizes.
    - Returns metrics as a float for automated tracking.
    """
    model.eval()
    device = next(model.parameters()).device
    total_rec_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for x, _ in dataloader:
            x = x.to(device)
            x_hat, _, _ = model(x)            
            loss = reconstruction_loss(x_hat, x) 
            total_rec_loss += loss.item() * x.size(0)
            total_samples += x.size(0)
            
    avg_rec = total_rec_loss / total_samples
    
    print(f"--- Fidelity Report {title_suffix} ---")
    print(f"Avg Reconstruction Error: {avg_rec:.6f}")
    
    return avg_rec

In [ ]:
test_reconstruction_fidelity(ae_model,  cifar10_test_loader, title_suffix="AE")
test_reconstruction_fidelity(vae_model, cifar10_test_loader, title_suffix="VAE")


## Synthetic OOD Stress Test

This test evaluates how an Autoencoder (AE) or Variational Autoencoder (VAE) handles **out-of-distribution (OOD)** inputs by using clean geometric patterns instead of natural CIFAR-10 images.

Because these shapes differ from the training data, reconstructions reveal what assumptions the model has learned.

- **Edge vs. Texture:** Difficulty reconstructing sharp shapes suggests the model learned natural image statistics rather than general geometry.
- **Latent Space Robustness:**  
  - **AE:** Often attempts direct reconstruction but may produce noise or artifacts.  
  - **VAE:** May reshape unusual inputs into familiar object-like forms due to latent regularization.
- **Average Effect:** Blurry gray or brown outputs suggest underfitting or an overly restrictive bottleneck, where the model defaults to dataset averages.

This test helps determine whether the model learned meaningful reusable structure or only the visual bias of its training set. It is especially useful for comparing AE vs. VAE generalization.

In [ ]:
plain_imgs = torch.zeros(4, 3, 32, 32)

# Single color channel
plain_imgs[1, 0] = 1
# Checkboard pattern
plain_imgs[2, :, :16, :16] = 1
plain_imgs[2, :, 16:, 16:] = -1
# Color progression
xx, yy = torch.meshgrid(torch.linspace(-1, 1, 32), torch.linspace(-1, 1, 32), indexing="xy")
plain_imgs[3, 0, :, :] = xx
plain_imgs[3, 1, :, :] = yy
visualize_reconstructions(ae_model,  plain_imgs, title_suffix="AE")


In [ ]:
visualize_reconstructions(vae_model, plain_imgs, title_suffix="VAE")

## Pure Noise Stress Test

This test feeds the model uniform random noise (maximum-entropy input with no structure) to evaluate how an Autoencoder (AE) or Variational Autoencoder (VAE) behaves far outside its training distribution.

It helps reveal:

- reliance on learned training priors  
- behavior under extreme out-of-distribution input  
- structure (or lack of structure) in the latent space  
- whether the model hallucinates meaningful patterns  

This test shows whether the model:

- preserves randomness,
- forces structure onto noise,
- or collapses to an average output,

making it useful for analyzing robustness and latent space behavior under extreme conditions.

In [ ]:
rand_imgs = torch.rand(8, 3, 32, 32) * 2 - 1
visualize_reconstructions(ae_model,  rand_imgs, title_suffix="AE")
visualize_reconstructions(vae_model, rand_imgs, title_suffix="VAE")

## Direct Latent Space Probing

This test bypasses the Encoder and feeds predefined latent vectors directly into the Decoder to see what images the model generates from specific points in latent space.

It treats the latent space as a learned “map” and asks what each coordinate corresponds to visually.

It evaluates:

- how structured and continuous the latent space is  
- whether unused regions produce meaningful outputs  
- how smooth transitions are between latent points  
- the overall generative quality of the Decoder  

A good latent space should behave smoothly:

- small changes in latent vectors → small changes in output images  
- interpolation between vectors → gradual visual transitions  

If this is not true, the latent space is likely poorly organized.

- **AE:**  
  Often produces noisy or unstable images from random latent vectors because its latent space is not well-regularized or continuous.

- **VAE:**  
  Produces more coherent and meaningful “sample-like” images because its latent space is structured around a smooth probabilistic prior.

This is a direct test of the latent space itself, independent of real input images. 

It reveals:

- how well the model organizes concepts internally  
- how usable the latent space is for generation  
- and how different AE and VAE designs affect representational structure

In [ ]:
def visualize_latent_reconstructions(model, latent_vectors, title_suffix=""):
    """
    Improved Latent Decoder Visualizer.
    - Handles device management (CPU/GPU) automatically.
    - Adds input type checking (converts lists/arrays to Tensors).
    - Optimizes grid layout and labeling.
    """
    model.eval()
    device = next(model.parameters()).device
    
    # 1. Ensure input is a Torch Tensor and move it to the model's device
    if not isinstance(latent_vectors, torch.Tensor):
        latent_vectors = torch.tensor(latent_vectors, dtype=torch.float32)
    latent_vectors = latent_vectors.to(device)

    # 2. Reconstruct
    with torch.no_grad():
        imgs = model.decoder(latent_vectors).cpu()
    
    # 3. Calculate an optimal grid size (square-ish layout)
    n_imgs = imgs.size(0)

    # 4. Create the visual grid
    grid = torchvision.utils.make_grid(
        imgs, 
        nrow=8, 
        normalize=True, 
        value_range=(-1, 1), 
        padding=2,
        pad_value=0.5  # Neutral gray padding
    )
    
    # 5. Plotting
    plt.figure(figsize=(10, 10))
    plt.imshow(grid.permute(1, 2, 0))
    plt.axis("off")
    
    plt.title(f"Decoder Samples {title_suffix}", fontsize=14)
    plt.tight_layout()
    plt.show()


latent_vectors = torch.randn(8, ae_model.latent_dim, device=device)
visualize_latent_reconstructions(ae_model, latent_vectors, title_suffix="AE")

visualize_latent_reconstructions(vae_model, latent_vectors, title_suffix="VAE")



In [ ]:
def random_sampling(model, n=8, title_suffix=""):
    latent_dim = model.latent_dim
    model.eval()
    with torch.no_grad():
        z = torch.randn(n, latent_dim).to(device)
        imgs = model.decoder(z).cpu()
    grid = torchvision.utils.make_grid(imgs, nrow=8, normalize=True, value_range=(-1, 1), pad_value=0.5)
    grid = grid.permute(1, 2, 0)
    plt.figure(figsize=(12, 6))
    plt.imshow(grid)
    plt.axis("off")
    plt.title(f"Reconstructed {title_suffix}")
    plt.show()

random_sampling(ae_model,  title_suffix="AE")
random_sampling(vae_model, title_suffix="VAE")


In [ ]:
def test_random_sampling(model, num_samples=16, title_suffix=""):
    """
    Samples directly from a standard normal distribution.
    VAE should produce recognizable objects; AE will likely produce noise.
    """
    model.eval()
    device = get_device()
    with torch.no_grad():
        z = torch.randn(num_samples, model.latent_dim).to(device)
        samples = model.decoder(z)
        grid = torchvision.utils.make_grid(samples, nrow=8, normalize=True)
        plt.figure(figsize=(12, 8))
        plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
        plt.title(f"Random Sampling {title_suffix}")
        plt.axis('off')
        plt.show()
    

In [ ]:
# Compare generation
test_random_sampling(ae_model,  title_suffix="AE")
test_random_sampling(vae_model, title_suffix="VAE")

## Latent Space Interpolation Test

This test visualizes how a model transitions between two images by linearly interpolating their latent vectors and decoding the intermediate points.

It checks whether the latent space is smooth and continuous or fragmented.

Two images are encoded into latent vectors, then blended using:

$
z_t = (1 - t)z_1 + tz_2
$

This creates a sequence of “morphing” images between the two inputs.

- **AE:** often produces noisy or unstable transitions  
- **VAE:** typically shows smoother, more gradual morphing due to latent regularization  


This test evaluates latent space smoothness by observing how well the model can “walk” between concepts rather than just encode isolated points.


In [ ]:
def show_interpolation(model, x1, x2, steps=10, title_suffix=""):
    """
    Improved Latent Interpolation.
    - Vectorized: Decodes all steps in a single batch.
    - Robust: Automatically handles VAE (mu/logvar) vs AE (z) outputs.
    - Clean: Highlights the 'Start' and 'End' images.
    """
    model.eval()
    device = next(model.parameters()).device
    
    # 1. Prepare inputs and move to device
    x1, x2 = x1.unsqueeze(0).to(device), x2.unsqueeze(0).to(device)

    with torch.no_grad():
        # 2. Get latent representations (Handling VAE/AE flexibility)
        out1 = model.encoder(x1)
        out2 = model.encoder(x2)
        z1 = out1[0] if isinstance(out1, (tuple, list)) else out1
        z2 = out2[0] if isinstance(out2, (tuple, list)) else out2

        # 3. Vectorized Interpolation: Calculate all z-steps at once
        # t is a column vector of shape (steps, 1)
        t = torch.linspace(0, 1, steps).view(-1, 1).to(device)
        # Linear combination: (1-t)*z1 + t*z2
        z_interp = (1 - t) * z1 + t * z2

        # 4. Single-pass decoding
        imgs = model.decoder(z_interp).cpu()

    # 5. Visualization
    grid = torchvision.utils.make_grid(
        imgs, 
        nrow=steps, 
        normalize=True, 
        value_range=(-1, 1), 
        padding=2
    )
    
    plt.figure(figsize=(steps * 1.5, 3))
    plt.imshow(grid.permute(1, 2, 0))
    
    plt.title(f"Latent Morphing {title_suffix} ({steps} steps)\nStart Image → End Image")
    plt.axis("off")
    plt.show()


# Example:
x1, _ = cifar10_test_set[0]
x2, _ = cifar10_test_set[1]

show_interpolation(ae_model,  x1, x2, title_suffix="AE")
show_interpolation(vae_model, x1, x2, title_suffix="VAE")

In [ ]:
def test_interpolation(model, dataloader, steps=10, title_suffix=""):
    """
    Improved Latent Space Interpolation.
    - Vectorized: Decodes all steps in one forward pass.
    - Robust: Handles VAE (mu, logvar) and AE return types.
    - Contextual: Displays the interpolation in a single, high-res row.
    """
    model.eval()
    device = next(model.parameters()).device
    
    # 1. Grab two random images from the loader
    it = iter(dataloader)
    x, _ = next(it)
    if x.size(0) < 2:
        raise ValueError("Batch size must be at least 2 to interpolate.")
    
    x1, x2 = x[0:1].to(device), x[1:2].to(device)
    
    with torch.no_grad():
        # 2. Flexible Encoding (Handles AE vs VAE automatically)
        out1 = model.encoder(x1)
        out2 = model.encoder(x2)
        z1 = out1[0] if isinstance(out1, (tuple, list)) else out1
        z2 = out2[0] if isinstance(out2, (tuple, list)) else out2
        
        # 3. Vectorized Linear Interpolation (LERP)
        # Reshape alphas to (steps, 1) so it broadcasts across latent_dim
        alphas = torch.linspace(0, 1, steps, device=device).view(-1, 1)
        z_interp = z1 * (1 - alphas) + z2 * alphas  # Result shape: (steps, latent_dim)
        
        # 4. Decodes everything in one batch (MUCH faster)
        imgs_interp = model.decoder(z_interp).cpu()
        
    # 5. Visual refinement
    grid = torchvision.utils.make_grid(
        imgs_interp, 
        nrow=steps, 
        normalize=True, 
        value_range=(-1, 1), 
        padding=2,
        pad_value=0.5
    )
    
    plt.figure(figsize=(max(steps, 12), 4))
    plt.imshow(grid.permute(1, 2, 0))
    
    plt.title(f"Latent Manifold: {title_suffix} (Continuous Transition)", fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
test_interpolation(ae_model,  cifar10_test_loader, title_suffix="AE")
test_interpolation(vae_model, cifar10_test_loader, title_suffix="VAE")

In [ ]:
def get_interpolation_grid(model, img1, img2, n_steps=10):
    """
    Performs vectorized latent interpolation between two images.
    Returns a single batch of interpolated images.
    """
    model.eval()
    device = next(model.parameters()).device
    
    # 1. Prepare images (ensure they have batch dim)
    img1, img2 = img1.to(device), img2.to(device)
    if img1.dim() == 3: img1 = img1.unsqueeze(0)
    if img2.dim() == 3: img2 = img2.unsqueeze(0)

    with torch.no_grad():
        # 2. Get latent vectors (handling both AE/VAE return styles)
        out1 = model.encoder(img1)
        out2 = model.encoder(img2)
        z1 = out1[0] if isinstance(out1, (tuple, list)) else out1
        z2 = out2[0] if isinstance(out2, (tuple, list)) else out2

        # 3. Vectorized Interpolation
        # Create a batch of lambdas: shape (n_steps, 1)
        lambdas = torch.linspace(0, 1, n_steps, device=device).view(-1, 1)
        
        # Calculate all interpolated latents at once via broadcasting
        # Shape: (n_steps, latent_dim)
        inter_latents = lambdas * z1 + (1 - lambdas) * z2

        # 4. Decode the entire batch in one go
        inter_images = model.decoder(inter_latents).cpu()
    
    return inter_images

def plot_interpolation(imgs, n_steps=10, title_suffix=""):
    grid = torchvision.utils.make_grid(imgs, nrow=n_steps, normalize=True, value_range=(-1,1))
    plt.figure(figsize=(15, 3))
    plt.imshow(grid.permute(1, 2, 0))
    plt.title(f"Interpolation: {title_suffix} (Class A -> Class B)")
    plt.axis("off")
    plt.show()

# --- Execution ---

# Picking specific images from your 'classes' list
# (Using your existing sorted 'classes' list logic)
# sort part of test set by digit
classes = [[] for _ in range(10)]
for img_batch, label_batch in cifar10_test_loader:
    for i in range(img_batch.size(0)):
        classes[label_batch[i]].append(img_batch[i:i+1])
    if sum(len(d) for d in classes) >= 1000:
        break;
start_img = classes[7][1] # Horse
end_img = classes[1][7]   # Automobile
steps = 10

# Process AE
ae_interp = get_interpolation_grid(ae_model, start_img, end_img, n_steps=steps)
plot_interpolation(ae_interp, steps, "Autoencoder")

# Process VAE
vae_interp = get_interpolation_grid(vae_model, start_img, end_img, n_steps=steps)
plot_interpolation(vae_interp, steps, "VAE")

## Latent Dimension Traversal (Disentanglement Study)

This test examines what individual dimensions in the latent space control by varying one dimension at a time while keeping all others fixed.

It checks whether the latent space is **disentangled**, meaning each dimension controls a distinct and interpretable feature.

Instead of moving between images, this test “turns one knob” in the latent vector:

- All dimensions are set to zero  
- One dimension is varied from −3 to +3  
- The decoder shows how output changes along that axis  

Each latent dimension may correspond to different features such as:

- object rotation  
- size or scale  
- color or texture changes  
- semantic shifts (e.g., different object types)  

A meaningful dimension produces smooth, consistent visual changes.

- **AE:** latent dimensions are often entangled or inconsistent, so changes may be random or meaningless  
- **VAE:** more likely to show smooth and structured changes due to regularized latent space (especially in β-VAEs)  

This test evaluates whether individual latent dimensions correspond to interpretable features, helping measure how disentangled and meaningful the learned representation is.

In [ ]:
def test_latent_traversal(model, dim_to_traverse=0, range_val=3.0, steps=12, title_prefix=""):
    """
    Improved Latent Traversal Visualizer.
    - Automatically detects model device.
    - Supports custom normalization ranges.
    - Labels the traversal limits for better interpretability.
    """
    model.eval()
    # Get device directly from model parameters (more robust)
    device = next(model.parameters()).device
    
    # 1. Create a batch of latent vectors at the origin (0,0,...)
    z = torch.zeros(steps, model.latent_dim).to(device)
    
    # 2. Linearly vary only the chosen dimension
    traversal = torch.linspace(-range_val, range_val, steps).to(device)
    z[:, dim_to_traverse] = traversal
    
    with torch.no_grad():
        # 3. Decode the entire traversal batch at once
        gen_images = model.decoder(z).cpu()
        
        # 4. Create the grid
        # value_range=(-1, 1) matches CIFAR-10 normalization
        grid = torchvision.utils.make_grid(
            gen_images, 
            nrow=steps, 
            normalize=True, 
            value_range=(-1, 1), 
            padding=2
        )
        
    # 5. Visualization
    plt.figure(figsize=(max(steps, 10), 2.5))
    plt.imshow(grid.permute(1, 2, 0))
    
    # Labeling extremes to help interpret the "knob" sensitivity
    plt.title(f"{title_prefix} Dimension {dim_to_traverse} Traversal: {range_val} to {-range_val}")
    plt.xlabel(f"Varying index {dim_to_traverse} while holding others at 0")
    plt.xticks([])
    plt.yticks([])
    plt.axis('on') # Turning axis back on briefly to label extremes
    plt.gca().set_yticks([])
    
    plt.show()

In [ ]:
test_latent_traversal(ae_model, title_prefix="AE")
test_latent_traversal(vae_model, title_prefix="VAE")

# Latent Space Image Retrieval (Nearest Neighbor Search)

This test evaluates how well a learned latent space encodes visual similarity by using it for nearest-neighbor image retrieval.

Each image is encoded into a latent vector, and similarity is measured using a distance metric (e.g., Euclidean or cosine distance). 

Given a query image, the closest latent vectors in the dataset are retrieved as matches.

If the representation is meaningful, visually or semantically similar images (e.g., same object class or structure) will lie close in latent space, even without using labels. 

This makes latent spaces useful as a form of “visual search engine” for unsupervised representation learning.

- **AE latent space**
  - Can preserve sharper local distinctions
  - Structure is not explicitly regularized
  - May produce less stable or uneven geometry

- **VAE latent space**
  - More structured and continuous manifold
  - Better suited for interpolation and generative sampling
  - May blur fine-grained class boundaries due to regularization


Neither approach is universally better for retrieval.

- **AE** → better at preserving local reconstruction-driven distinctions  
- **VAE** → better at producing a smooth, navigable latent manifold  

The effectiveness depends on the goal:

- clustering and separation → AE often stronger  
- interpolation and generative structure → VAE often better  

Latent space retrieval is a practical way to evaluate whether a model has learned meaningful representations beyond pixel-level reconstruction. 

Well-structured latent spaces naturally group similar visual concepts together, even without supervision.

In [ ]:
def embed_imgs(model, data_loader):
    """
    Encodes an entire dataset into latent vectors.
    - Robust: Handles VAE (mu, logvar) and AE return types.
    - Memory Efficient: Forces CPU offloading for images and embeddings.
    - Device Agnostic: Detects model location automatically.
    """
    img_list, embed_list = [], []
    model.eval()
    
    # Check device based on model parameters rather than a global function
    device = next(model.parameters()).device
    
    # Disable gradient calculation for speed and memory efficiency
    with torch.no_grad():
        for imgs, _ in tqdm(data_loader, desc="Encoding images", leave=False):
            # Move input to model device
            imgs_dev = imgs.to(device)
            
            # Handle AE/VAE flexibility: 
            # AE usually returns z; VAE usually returns (mu, logvar) or (z, mu, logvar)
            out = model.encoder(imgs_dev)
            z = out[0] if isinstance(out, (tuple, list)) else out
            
            # CRITICAL: Always move back to CPU before appending to a list.
            # This prevents a 'Memory Leak' on the GPU.
            embed_list.append(z.detach().cpu())
            
            # Store images on CPU as well to keep GPU memory for the model
            img_list.append(imgs.cpu())
    
    # Concatenate all batches into single tensors
    return { "images": torch.cat(img_list, dim=0), "embeddings": torch.cat(embed_list, dim=0) }

In [ ]:
ae_train_img_embeds  = embed_imgs(ae_model,  cifar10_train_loader)
ae_test_img_embeds   = embed_imgs(ae_model,  cifar10_test_loader)
vae_train_img_embeds = embed_imgs(vae_model, cifar10_train_loader)
vae_test_img_embeds  = embed_imgs(vae_model, cifar10_test_loader)


In [ ]:
def find_similar_images(query_img, query_embed, key_img_embeds, K=7, name=""):
    """
    Finds and displays images from a database that are most similar to a query image.
    Uses top-k selection for performance and refined grid visualization.
    """
    # 1. Distance Calculation
    # Ensure inputs are on the same device as the database
    device = key_img_embeds["embeddings"].device
    q_embed = query_embed.to(device)
    
    # torch.cdist is great, but for a single vector vs a matrix, 
    # a simple norm is often more readable and just as fast.
    dists = torch.norm(key_img_embeds["embeddings"] - q_embed, dim=1)
    
    # 2. Top-K Selection (Optimized)
    # Using topk is O(n log k) vs sort which is O(n log n). 
    # With large databases, this is a massive speedup.
    # largest=False gives us the smallest distances (closest neighbors).
    distances, indices = torch.topk(dists, k=K, largest=False)
    
    # 3. Image Preparation
    # Ensure query image has a batch dimension for concatenation
    q_img = query_img.cpu() if query_img.dim() == 3 else query_img[0].cpu()
    results = key_img_embeds["images"][indices].cpu()
    
    imgs_to_display = torch.cat([q_img.unsqueeze(0), results], dim=0)
    
    # 4. Visualization Logic
    padding = 2
    grid = torchvision.utils.make_grid(
        imgs_to_display, 
        nrow=K + 1, 
        normalize=True, 
        value_range=(-1, 1), 
        padding=padding
    )
    
    plt.figure(figsize=(max(K * 2, 10), 4))
    grid_np = grid.permute(1, 2, 0).numpy()
    plt.imshow(grid_np)
    
    # 5. Precise Highlighting
    # We calculate the exact pixel width of the first cell including half the padding
    # to make the red box look perfectly centered.
    _, total_w, _ = grid_np.shape
    cell_w = total_w // (K + 1)
    
    rect = plt.Rectangle(
        (0, 0), cell_w, grid_np.shape[0], 
        linewidth=4, edgecolor='r', facecolor='none'
    )
    plt.gca().add_patch(rect)
    
    plt.title(f"{name} Visual Search: Query (Red) vs Top {K} Results", fontsize=12)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot the closest images for the first N test images as example
for i in range(2):
    ae_query_img = ae_test_img_embeds["images"][i]
    ae_query_embed= ae_test_img_embeds["embeddings"][i]
    vae_query_img = ae_test_img_embeds["images"][i]
    vae_query_embed= ae_test_img_embeds["embeddings"][i]
    find_similar_images(ae_query_img,  ae_query_embed,  ae_train_img_embeds,  name="AE")
    find_similar_images(vae_query_img, vae_query_embed, vae_train_img_embeds, name="VAE") 

       


## Latent Space Dimensionality Reduction

This code compresses high-dimensional latent vectors (e.g., 64D) into 2D so the structure of the learned representation can be visualized.

The idea is to make the model’s internal representation interpretable by showing how it organizes images in space, and whether similar images end up near each other or are randomly scattered.

Images are first encoded into latent vectors along with their labels, then projected into 2D using PCA, t-SNE, or UMAP. The result is a scatter plot where each point represents an image and colors indicate classes.

###### AE vs VAE

Autoencoders often produce fragmented or irregular clusters because their latent space is not explicitly structured. 

Variational Autoencoders tend to form smoother, more continuous distributions due to probabilistic regularization, often resembling a Gaussian shape.

Clear clusters indicate strong feature learning, while overlap or scattering suggests weak structure. 

In well-learned representations, semantically similar classes also tend to appear closer together in space.


This visualization shows whether the model has learned meaningful structure in an unsupervised way, organizing images by visual similarity rather than memorization.

In [ ]:
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

def collect_latents(model, loader, max_batches=None):
    model.eval()
    Z, Y = [], []
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for i, (x, y) in enumerate(tqdm(loader, desc=f"Extracting {getattr(model, 'name', 'latent')}", leave=False)):
            x = x.to(device)
            # Handle AE (returns z) vs VAE (returns z, logvar)
            out = model.encoder(x)
            z = out[0] if isinstance(out, (tuple, list)) else out
            
            Z.append(z.cpu())
            Y.append(y)
            if max_batches and i >= max_batches:
                break

    return torch.cat(Z).numpy(), torch.cat(Y).numpy()

def compute_projections(Z):
    # CRITICAL: Scale data so features with larger magnitudes don't dominate PCA/UMAP
    Z_scaled = StandardScaler().fit_transform(Z)
    
    return {
        "PCA": PCA(n_components=2).fit_transform(Z_scaled),
        "t-SNE": TSNE(n_components=2, init='pca', learning_rate='auto', random_state=42).fit_transform(Z_scaled),
        "UMAP": umap.UMAP(n_components=2, random_state=42).fit_transform(Z_scaled)
    }

def plot_latent_comparison(model_data_list):
    """
    model_data_list: List of tuples [(Z, Y, model_name), ...]
    """
    methods = ["PCA", "t-SNE", "UMAP"]
    n_models = len(model_data_list)
    fig, axes = plt.subplots(n_models, len(methods), figsize=(18, 5 * n_models))
    
    # Ensure axes is always 2D even if only one model is passed
    if n_models == 1: axes = axes[None, :]

    for i, (Z, Y, name) in enumerate(model_data_list):
        projs = compute_projections(Z)
        for j, method in enumerate(methods):
            ax = axes[i, j]
            sc = ax.scatter(projs[method][:, 0], projs[method][:, 1], 
                            c=Y, s=2, cmap="tab10", alpha=0.7)
            ax.set_title(f"{name} - {method}", fontweight='bold')
            ax.set_xticks([]); ax.set_yticks([])

    # Shared colorbar for the whole figure
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    cbar_ax = fig.add_axes([0.15, 0.03, 0.7, 0.02])
    cbar = fig.colorbar(sc, cax=cbar_ax, orientation="horizontal")
    cbar.set_label("Class Label (CIFAR-10)")
    plt.show()

# --- Execution ---
Z_ae, Y_ae = collect_latents(ae_model, cifar10_test_loader, max_batches=20)
Z_vae, Y_vae = collect_latents(vae_model, cifar10_test_loader, max_batches=20)

plot_latent_comparison([
    (Z_ae, Y_ae, "Autoencoder"),
    (Z_vae, Y_vae, "VAE")
])

## Latent Norm Analysis

This test measures how far encoded images lie from the origin in latent space by computing their L2 norm. 

It provides a simple geometric view of how the model distributes representations.

It reveals whether the latent space is:

- tightly organized and regularized, or  
- spread out in an unconstrained way  

by looking at how concentrated or dispersed the encoded vectors are.

- **Autoencoder (AE):**  
  Produces widely spread and irregular latent norms because there is no constraint on where embeddings should lie. This leads to uneven structure, gaps in latent space, and poor alignment with any standard distribution.

- **Variational Autoencoder (VAE):**  
  Encourages latent vectors to follow a standard normal distribution, resulting in a tighter, more centered norm distribution. This makes the space more continuous and better suited for sampling.

For a latent space with dimension 64, a well-behaved VAE should have an average latent norm close to $\sqrt{64} \approx 8$, with relatively small variance.

Deviations from this indicate how strongly (or weakly) the latent space is aligned with the intended Gaussian structure.

This analysis shows how structured the latent space is geometrically and helps explain why VAEs support smooth sampling while AEs often do not. 

For a well-regularized VAE with latent vectors following $\mathcal{N}(0, I)$, the norms should approximately follow a $\chi(d)$ distribution, where $d$ is the latent dimension. 


Deviations from this shape indicate differences in latent space structure between models such as Autoencoders and VAEs.

It also provides a quick check of whether KL regularization is effectively shaping the representation.

This provides a direct geometric diagnostic of whether the latent space is properly Gaussian-like, overly spread, collapsed, or irregular.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi

def latent_norm_hist(model, loader, bins=50, log_scale=False, title_prefix=""):
    """
    Latent norm diagnostic comparing empirical L2 norms
    to the theoretical Chi distribution (||z|| for z ~ N(0, I)).
    """

    Z, _ = collect_latents(model, loader)

    if Z is None or len(Z) == 0:
        raise ValueError("No latent vectors found.")

    Z = np.asarray(Z)

    norms = np.linalg.norm(Z, axis=1)
    d = Z.shape[1]

    # --- Theoretical distribution (correct stats) ---
    theoretical_mean = chi.mean(d)
    theoretical_std = chi.std(d)

    # Robust plotting range (percentiles + theory)
    x_min = min(np.percentile(norms, 1), theoretical_mean * 0.5)
    x_max = max(np.percentile(norms, 99), theoretical_mean * 1.5)

    x = np.linspace(x_min, x_max, 500)
    chi_pdf = chi.pdf(x, df=d)

    # --- Plot ---
    plt.figure(figsize=(9, 5))

    plt.hist(
        norms,
        bins=bins,
        density=True,
        alpha=0.55,
        color="steelblue",
        edgecolor="white",
        label="Empirical (model)"
    )

    plt.plot(
        x,
        chi_pdf,
        "r--",
        linewidth=2.5,
        label=f"Theoretical χ(df={d})"
    )

    plt.axvline(norms.mean(), color="navy", linestyle="-",
                label=f"Empirical mean: {norms.mean():.2f}")

    plt.axvline(theoretical_mean, color="crimson", linestyle=":",
                label=f"Theoretical mean: {theoretical_mean:.2f}")

    plt.title(f"{title_prefix} Latent Norm Analysis")
    plt.xlabel(r"$||z||_2$")
    plt.ylabel("Density")

    if log_scale:
        plt.yscale("log")

    plt.grid(alpha=0.2)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # --- Diagnostics ---
    print(f"\n ========== {title_prefix} Latent Norm Diagnostics ==========")
    print(f"Latent dimension:        {d}")
    print(f"Empirical mean:          {norms.mean():.4f}")
    print(f"Theoretical mean (χ):    {theoretical_mean:.4f}")
    print(f"Std (empirical):         {norms.std():.4f}")
    print(f"Theoretical std (χ):     {theoretical_std:.4f}")
    print(f"Mean deviation:          {abs(norms.mean() - theoretical_mean):.4f}")

latent_norm_hist(ae_model,  cifar10_test_loader, title_prefix="AE")
latent_norm_hist(vae_model, cifar10_test_loader, title_prefix="VAE")

## Latent Density Check

This function measures how close each encoded point is to its nearest neighbor in latent space, giving a sense of how “dense” or “sparse” the learned representation is.

Each image becomes a point in latent space. 

This test checks how “social” those points are:

- **Very small distances:** points are tightly packed → possible memorization or over-clustering  
- **Moderate, consistent distances:** well-distributed manifold → smooth and continuous representation  
- **Highly variable distances:** uneven structure → gaps, isolated regions, or overfitting  

When you map an image into a latent vector, it becomes a single point in a high-dimensional room. 

This function calculates the distance to its absolute closest neighbor.

- If the distance is near zero: The points are "clumping."
  - The model is likely memorizing specific images rather than learning general concepts. This is like having a map where every city is just a single pinprick with nothing but a void in between.

- If the distance is moderate and consistent: The points are "filling" the space. 
  - This suggests a continuous manifold. It means the model has learned the "texture" of the data distribution, leaving no dead zones.

The space between points determines how well the model generalizes:

- **AE:** often forms uneven clusters with empty regions (“latent deserts”) and isolated memorized points  
- **VAE:** tends to create a more uniform and continuous space, though it can also collapse if over-regularized  

This test shows whether the latent space is a continuous, navigable landscape or a collection of isolated islands where interpolation breaks down.


In [ ]:
def latent_density_check(model, loader, k=2, title_suffix=""):
    """
    Checks how 'tightly packed' the latent space is.
    - Uses KDE for smoother visualization.
    - Adds median and mean markers.
    - Handles model-agnostic extraction.
    """
    # 1. Collect Latents
    Z, _ = collect_latents(model, loader)
    
    # 2. Compute Nearest Neighbors
    # k=2 because the 1st neighbor is always the point itself (dist=0)
    nbrs = NearestNeighbors(n_neighbors=k, algorithm='auto').fit(Z)
    distances, _ = nbrs.kneighbors(Z)
    nn_dist = distances[:, 1] # Distance to the actual nearest neighbor

    # 3. Visualization
    plt.figure(figsize=(10, 5))
    
    # Use a Kernel Density Estimate (KDE) for a clearer 'shape' of the density
    sns.histplot(nn_dist, bins=50, kde=True, color="teal", stat="density", alpha=0.4)
    
    # Statistical Markers
    mean_val = np.mean(nn_dist)
    median_val = np.median(nn_dist)
    plt.axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.4f}')
    plt.axvline(median_val, color='orange', linestyle='-', label=f'Median: {median_val:.4f}')

    # 4. Styling
    model_name = getattr(model, 'name', 'Model')
    plt.title(f"Latent Sparsity Check {title_suffix}", fontsize=14)
    plt.xlabel("Distance to Nearest Neighbor", fontsize=12)
    plt.ylabel("Density", fontsize=12)
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

    # 5. Diagnostic Output
    print(f"--- {title_suffix} Density Metrics ---")
    print(f"Mean NN Distance:   {mean_val:.6f}")
    print(f"Median NN Distance: {median_val:.6f}")
    print(f"Dist Variance:      {np.var(nn_dist):.6f}")


latent_density_check(ae_model,  cifar10_test_loader, title_suffix="AE")
latent_density_check(vae_model, cifar10_test_loader, title_suffix="VAE")

In [ ]:
def plot_explained_variance(Z_ae, Z_vae, threshold=0.90):
    """
    Improved Cumulative Explained Variance plot.
    - Adds a 90% variance threshold line.
    - Labels the 'elbow' point for both models.
    - Uses a cleaner layout.
    """
    plt.figure(figsize=(10, 6))
    
    for Z, label, color in [(Z_ae, "Autoencoder", "blue"), (Z_vae, "VAE", "orange")]:
        pca = PCA().fit(Z)
        cumsum = np.cumsum(pca.explained_variance_ratio_)
        
        # Find the number of components needed to reach the threshold
        d_eff = np.argmax(cumsum >= threshold) + 1
        
        plt.plot(cumsum, label=f"{label} (90% @ {d_eff} dims)", color=color, lw=2)
        plt.scatter(d_eff, cumsum[d_eff-1], color=color, s=50, zorder=5)

    # Aesthetics
    plt.axhline(y=threshold, color='red', linestyle='--', alpha=0.5)
    plt.title("Intrinsic Dimensionality: AE vs. VAE", fontsize=14)
    plt.xlabel("Number of Principal Components", fontsize=12)
    plt.ylabel("Cumulative Explained Variance", fontsize=12)
    plt.ylim(0, 1.05)
    plt.grid(alpha=0.3)
    plt.legend(loc="lower right")
    plt.show()

# Run the improved function
plot_explained_variance(Z_ae, Z_vae)


## Latent Distribution Analysis

This function evaluates how the latent space of an Autoencoder (AE) or Variational Autoencoder (VAE) is distributed compared to a standard normal distribution \( \mathcal{N}(0, I) \).

It helps diagnose whether the model has learned a well-structured latent space or one that is unbounded, collapsed, or unevenly distributed.

1. **Extracts latent vectors** from the encoder for the entire dataset  
2. **Compares global distribution** of all latent values against \( \mathcal{N}(0,1) \)  
3. **Analyzes each latent dimension individually** to detect collapse or imbalance  

###### Global distribution (left)
Shows whether latent values overall resemble a standard Gaussian distribution.

- Close match → well-regularized VAE  
- Wide / skewed distribution → typical AE behavior or weak KL regularization  

###### Per-dimension spread (right)
Shows how each latent dimension behaves individually.

- Tight or flat dimensions → “dead” or unused latent variables  
- Large variation across dimensions → uneven or unstructured representation  
- Consistent spread → healthy latent usage  

This test reveals whether the model is actually learning a **continuous, probabilistic latent space** (VAE) or just an unconstrained encoding (AE), and whether any latent dimensions are collapsing or underused.

- **AE:** latent values are unconstrained → irregular, non-Gaussian distribution  
- **VAE:** latent values are encouraged to follow \( \mathcal{N}(0, I) \) → more Gaussian-like structure  




In [ ]:
def test_latent_distribution(model, dataloader, num_dims_to_show=10, title_suffix=""):
    """
    Latent Distribution Analyzer (AE vs VAE diagnostic tool)

    This function evaluates how closely the learned latent space
    matches a standard normal distribution N(0, I).

    It provides:
    1. Global distribution shape (all latent values)
    2. Per-dimension analysis (detect collapse / dead dimensions)
    """

    model.eval()
    device = next(model.parameters()).device

    all_z = []

    # -------------------------
    # 1. Extract latent vectors
    # -------------------------
    with torch.no_grad():
        for x, _ in dataloader:
            x = x.to(device)

            z = model.encoder(x)

            # Handle both AE and VAE-style outputs
            if isinstance(z, (tuple, list)):
                z = z[0]

            all_z.append(z.cpu())

    all_z = torch.cat(all_z, dim=0).numpy()  # [N, latent_dim]
    latent_dim = all_z.shape[1]

    # -------------------------
    # 2. Visualization
    # -------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # ---- Global distribution ----
    flat = all_z.flatten()

    ax1.hist(flat, bins=100, density=True, alpha=0.6,
             color="steelblue", label="Empirical latent values")

    x = np.linspace(-5, 5, 300)
    ax1.plot(x, norm.pdf(x, 0, 1), "r--", lw=2,
             label=r"Ideal: $\mathcal{N}$(0,1)")

    ax1.set_title(f"Latent Value Distribution {title_suffix}")
    ax1.set_xlabel("Latent value")
    ax1.set_ylabel("Density")
    ax1.legend()
    ax1.grid(alpha=0.2)

    # ---- Per-dimension spread ----
    subset = all_z[:, :min(num_dims_to_show, latent_dim)]

    ax2.boxplot(subset, vert=False)
    ax2.axvline(0, color="red", linestyle="--", alpha=0.4)

    ax2.set_title(f"Per-Dimension Latent Spread (first {subset.shape[1]})")
    ax2.set_xlabel("Latent value")
    ax2.set_ylabel("Dimension")

    plt.tight_layout()
    plt.show()

    # -------------------------
    # 3. Summary statistics
    # -------------------------
    print(f"\n--- {title_suffix} Latent Distribution Summary ---")
    print(f"Latent dimension: {latent_dim}")
    print(f"Mean: {all_z.mean():.4f} (target = 0)")
    print(f"Std:  {all_z.std():.4f} (target = 1)")

In [ ]:
test_latent_distribution(ae_model,  cifar10_test_loader, title_suffix="AE")
test_latent_distribution(vae_model, cifar10_test_loader, title_suffix="VAE")


# Latent Space Analysis: AE vs VAE (Interpretation Guide)

This document provides a formal interpretation of the latent space diagnostic metrics used to compare Autoencoders (AE) and Variational Autoencoders (VAE).

### 1. Dataset and Model Configuration

###### samples

- Number of latent vectors evaluated.
- Ensures a statistically comparable evaluation.


###### latent_dim
- Dimensionality of the latent space.

- Both models operate under identical capacity constraints.

---

### 2. Spectral Structure of the Latent Space

These metrics are derived from the eigenvalue spectrum of the latent covariance matrix and describe how information is distributed across dimensions.

###### dims_90pct_var
- Number of principal components required to explain 90% of variance.

- Lower values indicate more concentrated information in fewer dimensions.

  - AE exhibits stronger compression and lower intrinsic dimensionality.
  - VAE distributes variance across more dimensions.


###### dims_95pct_var

- Number of components required to explain 95% of variance. 


###### participation_ratio

- Effective dimensionality based on eigenvalue concentration:

$
PR = \frac{(\sum \lambda_i)^2}{\sum \lambda_i^2}
$

- AE encodes information in a lower-dimensional subspace, whereas VAE utilizes nearly the full latent space.

###### effective_rank_entropy

- Entropy-based effective dimensionality.

- Measures how uniformly variance is distributed across dimensions.

  - AE: partially concentrated structure  
  - VAE: near-uniform variance distribution

###### isotropy_index

- Normalized entropy of eigenvalues (range: 0 to 1).
  - AE:  mildly anisotropic latent structure  
  - VAE: nearly perfectly isotropic latent space

---

###### eig_minmax_ratio

- Ratio between smallest and largest eigenvalues.

- Measures variance balance across directions.

  - AE: strong directional dominance  
  - VAE: relatively uniform variance distribution

### 3. Geometric Properties of the Latent Space

---

###### mean_norm

- Average Euclidean norm of latent vectors.
  - AE produces unconstrained latent magnitudes  
  - VAE enforces strong regularization toward a standard normal prior


###### std_norm

- Standard deviation of latent norms.
  - AE: heterogeneous encoding magnitudes  
  - VAE: tightly controlled latent scale

###### mean_nn_dist

- Average nearest-neighbor distance in latent space.
  - AE: sparse latent distribution  
  - VAE: dense latent distribution

###### mean_nn_dist_normalized

- Nearest-neighbor distance after normalization.

- Removes scale effects to assess intrinsic local structure.

  - VAE: more uniform local geometry  
  - AE: more structured clustering behavior


### 4. Statistical Properties of Latent Dimensions


###### mean_abs_skew

- Average absolute skewness of latent dimensions.
 
  - AE: slightly asymmetric marginal distributions  
  - VAE: closer to Gaussian symmetry

###### mean_abs_kurtosis

- Average absolute kurtosis (Gaussian reference ≈ 3).

  - AE: heavier-tailed distributions  
  - VAE: closer to Gaussian behavior

###### active_dims

- Number of dimensions with significant variance.

- Both models utilize the full latent dimensionality without collapse.

### 5. Similarity Structure of Latent Representations

###### mean_cosine_similarity

- Average cosine similarity between randomly sampled latent vectors.

  - AE: presence of structured correlations between latent codes  
  - VAE: near-random isotropic structure

###### std_cosine_similarity

- Standard deviation of cosine similarity values.

  - AE: more heterogeneous relational structure  
  - VAE: more uniform angular distribution

### Overall Interpretation

###### Autoencoder (AE)

- AE learns a compressed, structured representation of the data manifold and prioritizes representational efficiency and geometric structure.
  - Lower effective dimensionality
  - Strong anisotropy in latent space
  - Structured geometric manifold
  - Higher latent norms
  - Heavier-tailed distributions
  - Strong inter-sample correlations

###### Variational Autoencoder (VAE)

- VAE enforces a strongly regularized, approximately Gaussian latent space and prioritizes distributional regularity and smooth latent space geometry.
  - Near full-rank latent usage
  - Strong isotropy
  - Gaussian-like distributional structure
  - Low latent variance heterogeneity
  - Weak inter-sample correlation

| Property | AE | VAE |
|----------|----|-----|
| Structure | High | Low |
| Compression | Strong | Weak |
| Isotropy | Moderate | Very high |
| Gaussianity | Low | High |
| Manifold structure | Present | Weak |
| Distribution type | Structured | Near-Gaussian |

These results reflect the fundamental trade-off between reconstruction-driven learning (AE) and prior-constrained probabilistic modeling (VAE).

In [ ]:
# ==========================================================
# AE vs VAE Latent Space Diagnostic Suite
# ==========================================================

# -----------------------------
# constants
# -----------------------------
EPS = 1e-8
NN_K = 2
COSINE_SAMPLES = 10000
VAR_THRESHOLD_RATIO = 0.01


# -----------------------------
# utils
# -----------------------------
def participation_ratio(eigs):
    """
    Standard participation ratio (effective number of components).
    """
    return (eigs.sum() ** 2) / (np.dot(eigs, eigs) + EPS)


def mean_nn_distance(X):
    """
    Mean distance to nearest neighbor (excluding self).
    """
    nbrs = NearestNeighbors(n_neighbors=NN_K).fit(X)
    dists, _ = nbrs.kneighbors(X)
    return np.mean(dists[:, 1])


def sample_cosine_similarity(X, n_pairs=COSINE_SAMPLES, seed=0):
    """
    Monte Carlo estimate of cosine similarity distribution.
    """
    rng = np.random.default_rng(seed)
    N = len(X)

    i = rng.integers(0, N, n_pairs)
    j = rng.integers(0, N, n_pairs)

    Xi = X[i]
    Xj = X[j]

    dot = np.sum(Xi * Xj, axis=1)
    norm = (np.linalg.norm(Xi, axis=1) * np.linalg.norm(Xj, axis=1)) + EPS

    return dot / norm


def entropy_effective_rank(eigs):
    """
    Exponential of Shannon entropy of normalized eigenvalues.
    """
    p = eigs / (eigs.sum() + EPS)
    entropy = -np.sum(p * np.log(p + EPS))
    return np.exp(entropy)


def isotropy_index(eigs):
    """
    Normalized entropy (0 = collapsed, 1 = perfectly isotropic).
    """
    p = eigs / (eigs.sum() + EPS)
    entropy = -np.sum(p * np.log(p + EPS))
    return entropy / (np.log(len(eigs) + EPS))


# -----------------------------
# covariance / spectral metrics
# -----------------------------
def covariance_metrics(Z):
    """
    Spectral structure of latent space via SVD.
    """
    Zc = Z - Z.mean(axis=0, keepdims=True)

    # stable covariance spectrum via SVD
    _, S, _ = np.linalg.svd(Zc, full_matrices=False)
    eigs = (S ** 2) / (len(Z) - 1)
    eigs = np.sort(eigs)[::-1]

    p = eigs / (eigs.sum() + EPS)
    csum = np.cumsum(p)

    pr = participation_ratio(eigs)
    er = entropy_effective_rank(eigs)
    iso = isotropy_index(eigs)

    return {
        "dims_90pct_var": np.searchsorted(csum, 0.90) + 1,
        "dims_95pct_var": np.searchsorted(csum, 0.95) + 1,
        "participation_ratio": pr,
        "effective_rank_entropy": er,
        "isotropy_index": iso,
        "eig_minmax_ratio": eigs[-1] / (eigs[0] + EPS),
    }


# -----------------------------
# geometry metrics
# -----------------------------
def geometry_metrics(Z):
    norms = np.linalg.norm(Z, axis=1)

    nn = mean_nn_distance(Z)

    Zn = Z / (norms[:, None] + EPS)
    nn_norm = mean_nn_distance(Zn)

    return {
        "mean_norm": norms.mean(),
        "std_norm": norms.std(),
        "mean_nn_dist": nn,
        "mean_nn_dist_normalized": nn_norm,
    }


# -----------------------------
# statistical metrics
# -----------------------------
def statistical_metrics(Z):
    Zs = (Z - Z.mean(axis=0)) / (Z.std(axis=0) + EPS)

    return {
        "mean_abs_skew": np.mean(np.abs(skew(Zs, axis=0))),
        "mean_abs_kurtosis": np.mean(np.abs(kurtosis(Zs, axis=0, fisher=False))),
        "active_dims": np.sum(
            Z.var(axis=0) > VAR_THRESHOLD_RATIO * Z.var(axis=0).max()
        ),
    }


# -----------------------------
# similarity metrics
# -----------------------------
def similarity_metrics(Z):
    cos = sample_cosine_similarity(Z)
    return {
        "mean_cosine_similarity": cos.mean(),
        "std_cosine_similarity": cos.std(),
    }


# -----------------------------
# full pipeline
# -----------------------------
def latent_metrics(Z):
    """
    Complete latent-space diagnostic suite.
    """
    metrics = {
        "samples": Z.shape[0],
        "latent_dim": Z.shape[1],
    }

    metrics.update(covariance_metrics(Z))
    metrics.update(geometry_metrics(Z))
    metrics.update(statistical_metrics(Z))
    metrics.update(similarity_metrics(Z))

    return metrics


# -----------------------------
# run comparison
# -----------------------------
Z_ae, _ = collect_latents(ae_model, cifar10_test_loader)
Z_vae, _ = collect_latents(vae_model, cifar10_test_loader)

metrics_ae = latent_metrics(Z_ae)
metrics_vae = latent_metrics(Z_vae)

df = pd.DataFrame([metrics_ae, metrics_vae], index=["AE", "VAE"])

print(df.T.round(4))